# 35 -- Section 01e Step 5: back to the visible layer

Reconstructs the Rumble in the Bronx entry per channel, v+hat = sigmoid(s+ . W+^T + d+)
and v-hat = sigmoid(s- . W-^T + d-), using the exact procedure of section 01c's
Step 6 (same hidden-state sampling, same random draws), for both the 01c
(b = 0) and 01e (b = -1) input scenarios. Reuses notebook 31/32's exact
initial weights, biases and seeds. Does not touch the HTML.

## Part 0 -- Setup: scan notebooks/outputs, pick this notebook's number

In [1]:
from pathlib import Path
import json
import re

import pandas as pd
import torch

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
notebooks_dir = root / "notebooks"
data_dir = root / "data"
out_dir = root / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

nb_paths = sorted(p for p in notebooks_dir.glob("*.ipynb") if "step5_visible_reconstruction" not in p.stem)
num_re = re.compile(r"^(\d+)")
numbered = sorted(
    (int(m.group(1)), p) for p in nb_paths if (m := num_re.match(p.stem))
)
highest_num, highest_path = numbered[-1]
this_num = highest_num + 1
this_name = f"{this_num}_step5_visible_reconstruction.ipynb"
print(f"highest existing notebook: {highest_num:>3d}  ({highest_path.name})")
print(f"this notebook is:          {this_num:>3d}  ({this_name})")

highest existing notebook:  34  (34_step4_hidden_state.ipynb)
this notebook is:           35  (35_step5_visible_reconstruction.ipynb)


## Part 1 -- Reuse notebook 31/32's exact setup: data, movie, user, seeds, initial weights

In [2]:
train_path = data_dir / "train_hyperbolic.csv"
test_path = data_dir / "test_hyperbolic.csv"
results31_path = out_dir / "31_step8_results.json"
for p in (train_path, test_path, results31_path):
    assert p.exists(), f"Missing {p}"

train_hyp = pd.read_csv(train_path)
test_hyp = pd.read_csv(test_path)
assert len(train_hyp) == 492055 and len(test_hyp) == 123474, "not the section 03 split"

results31 = json.loads(results31_path.read_text())
REAL_SEED = results31["seeds"]["plus_seed_reused"]
MINUS_SEED = results31["seeds"]["minus_seed_trained"]
N_HIDDEN = int(results31["architecture_match"]["128 hidden units"].split("=")[1])
H1_IDX = 0
print(f"plus_seed={REAL_SEED}  minus_seed={MINUS_SEED}  N_HIDDEN={N_HIDDEN}")

all_users = sorted(pd.concat([train_hyp["userId"], test_hyp["userId"]]).unique())
all_movies = sorted(pd.concat([train_hyp["movieId"], test_hyp["movieId"]]).unique())
n_movies = len(all_movies)
movie_to_col = {m: j for j, m in enumerate(all_movies)}

movies = pd.read_csv(data_dir / "movie.csv")
rumble_row = movies[movies["title"].str.contains("Rumble in the Bronx", case=False, na=False)]
assert len(rumble_row) == 1
RUMBLE_ID = int(rumble_row.iloc[0]["movieId"])
RUMBLE_TITLE = rumble_row.iloc[0]["title"]
rumble_col = movie_to_col[RUMBLE_ID]

rumble4 = train_hyp[(train_hyp["movieId"] == RUMBLE_ID) & (train_hyp["rating"] == 4.0)].copy()
train_counts = train_hyp.groupby("userId").size()
rumble4["total_count"] = rumble4["userId"].map(train_counts)
rumble4 = rumble4.sort_values(["total_count", "userId"]).reset_index(drop=True)
median_pos = (len(rumble4) - 1) // 2
MEDIAN_USER = int(rumble4.iloc[median_pos]["userId"])
assert MEDIAN_USER == results31["worked_example_plus"]["user"]
print(f"movie: {RUMBLE_TITLE} (column={rumble_col})  user: {MEDIAN_USER}")

user_rows = train_hyp[train_hyp["userId"] == MEDIAN_USER]
v0_full = torch.zeros(n_movies)
for row in user_rows.itertuples():
    v0_full[movie_to_col[row.movieId]] = row.a
a_val = v0_full[rumble_col].item()
assert abs(a_val - results31["worked_example_plus"]["v0"]) < 1e-9

g_init_p = torch.Generator().manual_seed(REAL_SEED)
W_plus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_p) * 0.01
hbias_plus_init = torch.zeros(N_HIDDEN)
vbias_plus_init = torch.zeros(n_movies)  # d+ = 0, matching notebook 31's starting weights

g_init_m = torch.Generator().manual_seed(MINUS_SEED)
W_minus_init = torch.randn(n_movies, N_HIDDEN, generator=g_init_m) * 0.01
hbias_minus_init = torch.zeros(N_HIDDEN)
vbias_minus_init = torch.zeros(n_movies)  # d- = 0

d_plus_rumble = vbias_plus_init[rumble_col].item()
d_minus_rumble = vbias_minus_init[rumble_col].item()
print(f"d+[Rumble] = {d_plus_rumble}  d-[Rumble] = {d_minus_rumble}")

plus_seed=44  minus_seed=45  N_HIDDEN=128
movie: Rumble in the Bronx (Hont faan kui) (1995) (column=27)  user: 24127
d+[Rumble] = 0.0  d-[Rumble] = 0.0


## Part 2 -- Full 128-unit hidden state, sampled with the same seeds as notebook 31

Reconstruction needs the whole hidden vector (not just h1), since every
visible unit's reconstruction sums over all 128 hidden units. `b` only
changes the Rumble entry of `v`, so only that one entry differs between
the 01c and 01e visible vectors; the sampling generator is seeded exactly
as notebook 31 seeded it for this worked example.

In [3]:
b_val = -1.0
v_plus_01c = v0_full.clone()
v_minus_01c = v0_full.clone()
v_plus_01e = v0_full.clone(); v_plus_01e[rumble_col] = a_val + b_val
v_minus_01e = v0_full.clone(); v_minus_01e[rumble_col] = a_val - b_val

def hidden_sample(v_vec, W, hbias, seed):
    h_prob = torch.sigmoid(v_vec.unsqueeze(0) @ W + hbias)
    g = torch.Generator().manual_seed(seed)
    s = torch.bernoulli(h_prob, generator=g)
    return h_prob, s

h_plus_01c, s_plus_01c = hidden_sample(v_plus_01c, W_plus_init, hbias_plus_init, REAL_SEED)
h_minus_01c, s_minus_01c = hidden_sample(v_minus_01c, W_minus_init, hbias_minus_init, MINUS_SEED)
h_plus_01e, s_plus_01e = hidden_sample(v_plus_01e, W_plus_init, hbias_plus_init, REAL_SEED)
h_minus_01e, s_minus_01e = hidden_sample(v_minus_01e, W_minus_init, hbias_minus_init, MINUS_SEED)

print(f"h+_01c[h1] = {h_plus_01c[0,H1_IDX].item():.6f}  (check vs 31: {results31['worked_example_plus']['h0_1']:.6f})")
print(f"h-_01c[h1] = {h_minus_01c[0,H1_IDX].item():.6f}  (check vs 31: {results31['worked_example_minus']['h0_1']:.6f})")
print(f"s+_01c on-count = {int(s_plus_01c.sum().item())}/128   s-_01c on-count = {int(s_minus_01c.sum().item())}/128")
print(f"s+_01e on-count = {int(s_plus_01e.sum().item())}/128   s-_01e on-count = {int(s_minus_01e.sum().item())}/128")

h+_01c[h1] = 0.447416  (check vs 31: 0.447416)
h-_01c[h1] = 0.448838  (check vs 31: 0.448838)
s+_01c on-count = 55/128   s-_01c on-count = 69/128
s+_01e on-count = 55/128   s-_01e on-count = 69/128


## Part 3 -- Reconstruct the Rumble entry per channel: v-hat = sigmoid(s . W^T + d)

In [4]:
def reconstruct_rumble(s, W, vbias, col):
    z = (s @ W[col, :]).item() + vbias[col].item()
    return torch.sigmoid(torch.tensor(z)).item(), z

vhat_plus_01c, z_plus_01c = reconstruct_rumble(s_plus_01c, W_plus_init, vbias_plus_init, rumble_col)
vhat_minus_01c, z_minus_01c = reconstruct_rumble(s_minus_01c, W_minus_init, vbias_minus_init, rumble_col)
vhat_plus_01e, z_plus_01e = reconstruct_rumble(s_plus_01e, W_plus_init, vbias_plus_init, rumble_col)
vhat_minus_01e, z_minus_01e = reconstruct_rumble(s_minus_01e, W_minus_init, vbias_minus_init, rumble_col)

print(f"v+hat_01c = sigmoid({z_plus_01c:.4f}) = {vhat_plus_01c:.6f}  (check vs 31 Step 8 table: 0.4885)")
print(f"v-hat_01c = sigmoid({z_minus_01c:.4f}) = {vhat_minus_01c:.6f}  (check vs 31 Step 8 table: 0.4632)")
print(f"v+hat_01e = sigmoid({z_plus_01e:.4f}) = {vhat_plus_01e:.6f}")
print(f"v-hat_01e = sigmoid({z_minus_01e:.4f}) = {vhat_minus_01e:.6f}")

assert abs(vhat_plus_01c - 0.4885) < 5e-4, "01c plus reconstruction must match the page's Step 8 table"
assert abs(vhat_minus_01c - 0.4632) < 5e-4, "01c minus reconstruction must match the page's Step 8 table"
print("01c column matches the values already shown in 01c. OK.")

v+hat_01c = sigmoid(-0.0462) = 0.488463  (check vs 31 Step 8 table: 0.4885)
v-hat_01c = sigmoid(-0.1474) = 0.463210  (check vs 31 Step 8 table: 0.4632)
v+hat_01e = sigmoid(-0.0462) = 0.488463
v-hat_01e = sigmoid(-0.1474) = 0.463210
01c column matches the values already shown in 01c. OK.


## Part 4 -- Recombine into one hyperbolic number: a-hat, r-hat, b-hat

In [5]:
def combine(vhat_plus, vhat_minus):
    ahat = (vhat_plus + vhat_minus) / 2
    bhat = (vhat_plus - vhat_minus) / 2
    rhat = 4.5 * ahat + 0.5
    return ahat, bhat, rhat

ahat_01c, bhat_01c, rhat_01c = combine(vhat_plus_01c, vhat_minus_01c)
ahat_01e, bhat_01e, rhat_01e = combine(vhat_plus_01e, vhat_minus_01e)

print(f"01c: a-hat={ahat_01c:.6f}  r-hat={rhat_01c:.6f}  b-hat={bhat_01c:.6f}  (target b=0)")
print(f"01e: a-hat={ahat_01e:.6f}  r-hat={rhat_01e:.6f}  b-hat={bhat_01e:.6f}  (target b=-1)")

for bhat in (bhat_01c, bhat_01e):
    assert -0.5 < bhat < 0.5, "sigmoid keeps each channel in (0,1), so b-hat must stay in (-0.5, 0.5)"
print("both b-hat values confirmed inside (-0.5, 0.5) in code.")

01c: a-hat=0.475837  r-hat=2.641264  b-hat=0.012627  (target b=0)
01e: a-hat=0.475837  r-hat=2.641264  b-hat=0.012627  (target b=-1)
both b-hat values confirmed inside (-0.5, 0.5) in code.


## Part 5 -- Save for Step 5 of section 01e

In [6]:
results = {
    "notebook": this_name,
    "reused_from": "31_step8_minus_channel.ipynb, 32_step2_hidden_input.ipynb",
    "movie": RUMBLE_TITLE,
    "user": MEDIAN_USER,
    "hidden_unit": "h1 (index 0), full 128-unit state used for reconstruction",
    "d_plus_rumble": d_plus_rumble,
    "d_minus_rumble": d_minus_rumble,
    "vhat_plus": {"01c": vhat_plus_01c, "01e": vhat_plus_01e},
    "vhat_minus": {"01c": vhat_minus_01c, "01e": vhat_minus_01e},
    "ahat": {"01c": ahat_01c, "01e": ahat_01e},
    "rhat": {"01c": rhat_01c, "01e": rhat_01e},
    "bhat": {"01c": bhat_01c, "01e": bhat_01e},
    "target_b": {"01c": 0.0, "01e": -1.0},
}
results_path = out_dir / "35_step5_visible_reconstruction_results.json"
with results_path.open("w") as f:
    json.dump(results, f, indent=2)
print(f"saved {results_path}")
print(json.dumps(results, indent=2))

saved /Users/yixuan/Boltzmann Machine in Movie Lens/rbm-recsys/outputs/35_step5_visible_reconstruction_results.json
{
  "notebook": "35_step5_visible_reconstruction.ipynb",
  "reused_from": "31_step8_minus_channel.ipynb, 32_step2_hidden_input.ipynb",
  "movie": "Rumble in the Bronx (Hont faan kui) (1995)",
  "user": 24127,
  "hidden_unit": "h1 (index 0), full 128-unit state used for reconstruction",
  "d_plus_rumble": 0.0,
  "d_minus_rumble": 0.0,
  "vhat_plus": {
    "01c": 0.48846325278282166,
    "01e": 0.48846325278282166
  },
  "vhat_minus": {
    "01c": 0.4632098376750946,
    "01e": 0.4632098376750946
  },
  "ahat": {
    "01c": 0.47583654522895813,
    "01e": 0.47583654522895813
  },
  "rhat": {
    "01c": 2.6412644535303116,
    "01e": 2.6412644535303116
  },
  "bhat": {
    "01c": 0.012626707553863525,
    "01e": 0.012626707553863525
  },
  "target_b": {
    "01c": 0.0,
    "01e": -1.0
  }
}
